# 04 — Búsqueda de hiperparámetros: red neuronal (MLP)

Un MLP feed-forward (PyTorch). Buscamos arquitectura y **regularización**:

- Arquitectura: `hidden_dims` (capas y anchos).
- Regularización: `dropout`, `weight_decay` (L2 vía el optimizador). Además, cada
  modelo hace *early stopping* sobre un split interno de validación.
- Optimización: `lr`.

La red estandariza el target internamente. Búsqueda aleatoria con la misma CV
temporal; el test se reserva para el final. (Menos combinaciones que XGBoost porque
cada entrenamiento es más caro.)

**Nota:** con solo unos miles de filas, una red poco regularizada sobreajusta con
ganas (memoriza el rinde medio de cada depto en train y no generaliza al test, que
además cae en años *fuera* del rango de train). Por eso la grilla apunta a redes
chicas y regularización fuerte — el `dropout` alto es el que más mueve la aguja.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import LinearRegressor, XGBoostRegressor, NeuralNetRegressor

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


## La búsqueda

Búsqueda aleatoria de 14 combinaciones, optimizando RMSE de validación temporal.

In [ ]:
grid = {
    'hidden_dims':  [(16,), (32,), (64, 32)],
    'dropout':      [0.1, 0.3, 0.5],
    'weight_decay': [1e-3, 1e-2, 3e-2],
    'lr':           [1e-3, 3e-3],
}
tabla, best = ev.buscar(NeuralNetRegressor, grid, ds, metric='rmse',
                        n_iter=14, random_state=42, n_splits=4,
                        fixed={'max_epochs': 200, 'patience': 25, 'l1_lambda': 0.0})
print('Mejores hiperparámetros:', best)
tabla.head(10)

## Modelo final

Reentrenamos con los mejores hiperparámetros sobre todo el train y evaluamos en el
test. **Métricas del modelo final:**

In [ ]:
modelo = NeuralNetRegressor(**best, l1_lambda=0.0, max_epochs=250, patience=30,
                            random_state=42).fit(ds.X_train, ds.y_train)
pred = modelo.predict(ds.X_test)

m = ev.metricas(ds.y_test, pred)
print(f"MAE  = {m['mae']:.1f} kg/ha")
print(f"RMSE = {m['rmse']:.1f} kg/ha")
print(f"R²   = {m['r2']:.3f}")
print(f"MAPE = {m['mape']:.1f} %")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ev.plot_pred_vs_real(ds.y_test, pred, 'Red neuronal (final)', color=ev.C_NN, ax=axes[0])
ev.plot_residuos(ds.y_test, pred, 'Residuos', color=ev.C_NN, ax=axes[1])
h = modelo.history_
axes[2].plot(h['train_loss'], label='train', color=ev.C_LINEAR)
if h['val_loss']:
    axes[2].plot(h['val_loss'], label='val', color=ev.C_NN)
axes[2].set_title('Curva de entrenamiento (MSE, target escalado)')
axes[2].set_xlabel('época'); axes[2].legend()
plt.tight_layout(); plt.show()

## Ambos datasets + el mejor modelo del Componente A

Con la **configuración final ya elegida**, evaluamos en test:

1. dataset `base` (solo clima) vs `era5_ndvi` (clima + NDVI + ERA5), y
2. sobre el que mejor anduvo, tres estrategias que reusan el **mejor detector del
   Componente A** (el VAE `recon_prob`, que aprendió a representar el clima
   "normal"): concatenar su **espacio latente**, hacer la regresión **solo en el
   latente**, y agregar la categórica **`es_anomalo`** (su score umbralado).

El latente/score se computan en `latente.py` (y se cachean). *La primera corrida
entrena el VAE, así que tarda unos minutos.*

In [ ]:
tabla_ds, mejor = ev.comparar_datasets_y_latente(
    NeuralNetRegressor, best, CULTIVO, fixed=dict(l1_lambda=0.0, max_epochs=250, patience=30, random_state=42),
    vae_kwargs=dict(score_seeds=(42, 43, 44)))
print('Mejor dataset base:', mejor)
tabla_ds

## Conclusión

Con tan pocos miles de filas y features tabulares, la red neuronal compite pero no
suele destronar a un XGBoost bien tuneado. El notebook 05 compara las tres familias
más los baselines en igualdad de condiciones.